## 🔧 Reward Fix - Position Holding Threshold

**Problem Found:**
- No rewards for holding positions with small profits (< 1.0%)
- Example: UnrPnL% = 0.11% → 0.0000 reward (should be positive!)

**Fix Applied:**
- Lowered threshold: 1.0% → **0.05%** (20x more sensitive)
- Tightened drawdown penalty: -3.0% → **-2.0%** (earlier warning)
- Now rewards even small profitable holds (+0.1 base per step)

**Expected After Fix:**
- Step with 0.11% profit → reward ≈ 0.13 (0.11×0.3 + 0.1)
- Step with 0.16% profit → reward ≈ 0.15
- Encourages holding winners even during small fluctuations

In [ ]:
import numpy as np
import pandas as pd
from src.environments.simple_trading_env import SimpleTradingEnv
from src.utils.indicator_utils import add_indicators
import json

# Load data
symbol = 'BTCUSDT'
timeframe = '5m'
data_path = f'data/binance-{symbol}-{timeframe}.pkl'
df = pd.read_pickle(data_path)

# Create environment (no wrappers for direct testing)
env = SimpleTradingEnv(df)
test_data = env.data

print(f"\n{'='*70}")
print("Environment initialized - Ready for action testing")
print(f"{'='*70}")
print(f"Initial Balance: ${env.initial_balance:,.2f}")
print(f"Action Space: {env.action_space}")
print(f"  Action format: [direction, risk_reward_ratio, atr_multiplier]")
print(f"  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE")
print(f"  Risk-Reward: 0-9 (maps to 1.0x-10.0x)")
print(f"  ATR Multiplier: 0-9 (maps to 1.5-3.3)")
print(f"{'='*70}\n")

## Test 1: Random Actions

In [ ]:
np.random.seed(42)
obs, _ = env.reset()

steps_to_run = 100
history_records = []

print(f"Running {steps_to_run} steps with RANDOM actions...")
print(f"{'='*70}\n")

for step in range(steps_to_run):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)
df_history['direction'] = df_history['action'].apply(lambda x: x[0] if isinstance(x, np.ndarray) else x)

# Summary
print("RANDOM ACTION TEST SUMMARY")
print(f"{'='*70}")
print(f"Total Steps: {len(df_history)}")
print(f"Total Reward: {df_history['reward'].sum():.4f} | Avg/Step: {df_history['reward'].mean():.4f}")
print(f"Final Equity: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f}")

# Action distribution
action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
action_counts = df_history['direction'].value_counts().sort_index()
print(f"\nAction Distribution:")
for action_id, count in action_counts.items():
    print(f"  {action_names.get(action_id, action_id):>6}: {count:3d} ({count/len(df_history)*100:5.1f}%)")

# Reward by action
print(f"\nReward by Action:")
reward_summary = df_history.groupby('direction')['reward'].agg(['count', 'mean', 'sum']).round(4)
for idx, row in reward_summary.iterrows():
    print(f"  {action_names.get(idx, idx):>6}: count={row['count']:3.0f}, mean={row['mean']:7.4f}, sum={row['sum']:8.4f}")

print(f"{'='*70}\n")

## Test 2: Custom action sequence

In [ ]:
# Reset environment for sequence test
env.reset()


# start 288 
# trage from 441
# Define specific action sequence
action_sequence = (
    [[0, 0, 0]] * (444 - 288) +      # Wait 6 steps
    [[1, 0, 0]] +          # Enter Long
    [[0,0,0]] * 60     # Hold for 160 steps
)

print(f"Sequence: HOLD(2) → SHORT → HOLD(160) → CLOSE")
print(f"{'='*80}\n")

history_records = []
for step, action in enumerate(action_sequence):
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)

# Summary
print(f"SEQUENCE TEST SUMMARY")
print(f"{'='*80}")
print(f"Steps: {len(df_history)} | Total Reward: {df_history['reward'].sum():.4f}")
print(f"Initial: ${df_history['equity'].iloc[0]:,.2f} | Final: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f} ({(df_history['equity'].iloc[-1] / df_history['equity'].iloc[0] - 1) * 100:.2f}%)")
print(f"{'='*80}\n")

# Show ALL steps in the sequence
print("ALL STEPS IN SEQUENCE:")
print(f"{'Step':>4} | {'Action':>6} | {'Price':>10} | {'Reward':>8} | {'Equity':>10} | {'UnrPnL%':>8} | {'Note'}")
print(f"{'-'*80}")

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
prev_pos = 0

for i, row in df_history.iterrows():
    action_dir = row['action'][0] if isinstance(row['action'], (list, np.ndarray)) else row['action']
    action_name = action_names.get(action_dir, str(action_dir))
    pos_size = row['position_size']
    unrealized = row.get('unrealized_pnl', 0)
    used_bal = row.get('used_balance', 0)
    unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
    
    note = row['reason']
    print(f"{row['step']:4d} | {action_name:>6} | {row['current_price']} | {row['reward']:8.4f} | ${row['equity']:9.2f} | {unrealized_pct:7.2f}% | {note}")
    
    prev_pos = pos_size

print(f"{'-'*80}\n")

# Trade analysis - Show trade details across history
all_trades = history_records[-1].get('trades', [])
closed_trades = [t for t in all_trades if t.get('status') == 'CLOSED']

if closed_trades:
    print(f"TRADE ANALYSIS - All Closed Trades")
    print(f"{'='*80}")
    
    for trade_idx, trade in enumerate(closed_trades, 1):
        print(f"\nTrade #{trade_idx}")
        print(f"{'-'*80}")
        print(f"Direction: {'LONG' if trade['direction'] == 1 else 'SHORT'}")
        print(f"Entry Step: {trade['step_open']} | Exit Step: {trade['step_close']} | Duration: {trade.get('duration', 0)} steps")
        print(f"Entry: ${trade['entry_price']:,.2f} | Exit: ${trade.get('exit_price', 0):,.2f}")
        print(f"PnL: ${trade['pnl']:,.2f} ({trade['pnl_percent']*100:.2f}%) | Exit Reason: {trade['reason']}")
        
        # Show the trade progression across history steps
        step_open = trade['step_open']
        step_close = trade['step_close']
        
        print(f"\nTrade progression (steps {step_open} to {step_close}):")
        print(f"{'Step':>4} | {'Price':>8} | {'Action':>6} | {'Equity':>10} | {'UnrPnL':>8} | {'UnrPnL%':>8} | {'Reward':>8}")
        print(f"{'-'*70}")
        
        # Show relevant steps during this trade
        for i, record in enumerate(history_records):
            step = record['step']
            
            # Show steps within the trade range
            if step_open <= step <= step_close:
                action_dir = record['action'][0] if isinstance(record['action'], (list, np.ndarray)) else record['action']
                action_name = action_names.get(action_dir, str(action_dir))
                price = record.get('current_price', 0)
                equity = record['equity']
                unrealized = record.get('unrealized_pnl', 0)
                used_bal = record.get('used_balance', 0)
                unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
                reward = (record.get('reward', -1))
                
                marker = ""
                if step == step_open:
                    marker = " ← ENTRY"
                elif step == step_close:
                    marker = " ← EXIT"
                
                print(f"{step:4d} | ${price:7.2f} | {action_name:>6} | ${equity:9.2f} | ${unrealized:7.2f} | {unrealized_pct:7.2f}% |{reward:.4f} {marker}")
        
        print(f"{'-'*70}")
else:
    print("No closed trades found.")


## 🔄 Test with Fixed Reward Function

**Fix Applied:**
- Lowered threshold: 1.0% → **0.05%** (20x more sensitive)
- Tightened drawdown penalty: -3.0% → **-2.0%** 
- Now rewards based on **actual unrealized P&L %**, not just step count
- Rewards scale with profit: 0.05% → 0.1, 1% → 0.3, 3% → 0.9

**Expected After Fix:**
- Step with 1.40% profit → reward ≈ 0.51 (tanh(1.4×0.3) + 0.1)
- Step with 3.47% profit → reward ≈ 0.82 (tanh(3.47×0.3) + 0.1)
- Properly encourages holding winners!

In [ ]:
# Reload the environment to get the updated reward function
import importlib
import sys

# Remove cached modules
if 'src.environments.simple_trading_env' in sys.modules:
    del sys.modules['src.environments.simple_trading_env']

from src.environments.simple_trading_env import SimpleTradingEnv

# Recreate environment with fixed reward function
env = SimpleTradingEnv(df)
print("✅ Environment reloaded with fixed reward function!")

In [ ]:
# Re-run the same sequence with fixed rewards
env.reset()

action_sequence = (
    [[0, 0, 0]] * (444 - 288) +      # Wait to step 444
    [[1, 0, 0]] +                     # Enter Long
    [[0,0,0]] * 60                    # Hold for 60 steps
)

print(f"Sequence: HOLD → LONG → HOLD(60)")
print(f"{'='*80}\n")

history_records = []
for step, action in enumerate(action_sequence):
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)

# Summary
print(f"FIXED REWARD TEST SUMMARY")
print(f"{'='*80}")
print(f"Steps: {len(df_history)} | Total Reward: {df_history['reward'].sum():.4f}")
print(f"Initial: ${df_history['equity'].iloc[0]:,.2f} | Final: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f} ({(df_history['equity'].iloc[-1] / df_history['equity'].iloc[0] - 1) * 100:.2f}%)")
print(f"{'='*80}\n")

# Show ALL steps in the sequence
print("ALL STEPS WITH FIXED REWARDS:")
print(f"{'Step':>4} | {'Action':>6} | {'Price':>10} | {'Reward':>8} | {'Equity':>10} | {'UnrPnL%':>8}")
print(f"{'-'*80}")

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}

for i, row in df_history.iterrows():
    action_dir = row['action'][0] if isinstance(row['action'], (list, np.ndarray)) else row['action']
    action_name = action_names.get(action_dir, str(action_dir))
    unrealized = row.get('unrealized_pnl', 0)
    used_bal = row.get('used_balance', 0)
    unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
    
    print(f"{row['step']:4d} | {action_name:>6} | {row['current_price']:>10.2f} | {row['reward']:8.4f} | ${row['equity']:9.2f} | {unrealized_pct:7.2f}%")

print(f"{'-'*80}\n")

# Compare rewards
print("REWARD IMPROVEMENT ANALYSIS:")
print(f"{'='*80}")
print(f"Total reward: {df_history['reward'].sum():.4f}")
print(f"Average reward per step: {df_history['reward'].mean():.4f}")
print(f"Max reward in single step: {df_history['reward'].max():.4f}")
print(f"Steps with positive reward: {(df_history['reward'] > 0).sum()} / {len(df_history)}")
print(f"{'='*80}")